# Clipper Whisper Worker (Colab GPU)
Jalankan semua cell berurutan. Di cell terakhir, copy URL tunnel + token ke Settings Clipper.


In [ ]:
!pip install -q "faster-whisper==1.1.1" "av>=12,<19" "opencv-python-headless"
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
import av; print('av version:', av.__version__)
assert int(av.__version__.split('.')[0]) < 19, 'PyAV 19+ tidak kompatibel dengan faster-whisper 1.1.1; jalankan ulang install cell'


In [ ]:
# Download YuNet face detector model untuk face tracking di Colab.
import os, urllib.request
os.makedirs('/content/models', exist_ok=True)
url = 'https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx'
dst = '/content/models/face_detection_yunet_2023mar.onnx'
if not os.path.exists(dst):
    urllib.request.urlretrieve(url, dst)
print('model:', dst, os.path.exists(dst), os.path.getsize(dst) if os.path.exists(dst) else 0)
os.environ['FACE_MODEL_PATH'] = dst


In [ ]:
# Daftarkan lib cuBLAS/cuDNN dari pip ke dynamic loader (ldconfig).
# os.environ['LD_LIBRARY_PATH'] di Python tidak berpengaruh ke proses yang sudah jalan.
import os, glob, site, ctypes
libdirs = []
for sp in site.getsitepackages():
    libdirs += [d for d in glob.glob(sp + '/nvidia/*/lib') if os.path.isdir(d)]
conf = '/etc/ld.so.conf.d/nvidia-pip.conf'
with open(conf, 'w') as f:
    f.write('\n'.join(libdirs) + '\n')
os.system('ldconfig')
print(open(conf).read())
# preload ke proses kernel yang sedang jalan
for name in ('libcublas.so.12','libcublasLt.so.12','libcudnn.so.9','libcudnn_ops.so.9'):
    for d in libdirs + ['/usr/local/cuda/lib64']:
        pp = os.path.join(d, name)
        if os.path.exists(pp):
            try:
                ctypes.CDLL(pp, mode=ctypes.RTLD_GLOBAL)
                print('preloaded', pp)
            except OSError as e:
                print('skip', pp, e)
            break
ctypes.CDLL('libcublas.so.12')
print('cublas OK')


In [ ]:
import os, secrets
os.environ.setdefault('WORKER_TOKEN', os.getenv('WORKER_TOKEN') or secrets.token_urlsafe(24))
os.environ.setdefault('WHISPER_MODEL', 'large-v3-turbo')
os.environ.setdefault('WHISPER_COMPUTE', 'float16')
os.environ.setdefault('FACE_MODEL_PATH', '/content/models/face_detection_yunet_2023mar.onnx')
os.environ.setdefault('ALLOW_CPU_FALLBACK', '1')
open('/content/.clipper_token','w').write(os.environ['WORKER_TOKEN'])
import site, glob
libdirs = []
for sp in site.getsitepackages():
    libdirs += glob.glob(sp + '/nvidia/*/lib')
os.environ['LD_LIBRARY_PATH'] = ':'.join(libdirs + [os.environ.get('LD_LIBRARY_PATH','')])
print('='*60)
print('WORKER_TOKEN :', os.environ['WORKER_TOKEN'])
print('MODEL        :', os.environ['WHISPER_MODEL'])
print('av           :', __import__('av').__version__)
print('='*60)


In [ ]:
# Hotfix kompatibilitas: kalau Colab memaksa PyAV 19+, buang kwarg yang sudah dihapus.
import av
def _av_needs_patch() -> bool:
    try:
        return int(str(av.__version__).split('.')[0]) >= 19
    except Exception:
        try:
            import inspect
            return 'metadata_errors' not in inspect.signature(av.open).parameters
        except (ValueError, TypeError):
            return False
if _av_needs_patch():
    _orig_open = av.open
    def _open_compat(*a, **kw):
        for key in ('metadata_errors','metadata_encoding','stream_options'):
            kw.pop(key, None)
        return _orig_open(*a, **kw)
    av.open = _open_compat
    print('av patched:', av.__version__)
else:
    print('av OK, no patch needed:', av.__version__)


In [ ]:
# Opsional: login HF untuk download lebih cepat (hindari rate limit).
# from huggingface_hub import login; login()
print('HF login dilewati (opsional).')


In [ ]:
!wget -q https://raw.githubusercontent.com/Maulana2505/clipper-Remotion-renderer-/main/colab/worker.py -O /content/worker.py
import os
print('worker.py ready:', os.path.exists('/content/worker.py'))


In [ ]:
# Verifikasi CUDA + model + decode audio sebelum tunnel dijalankan
import os, ctranslate2, av, inspect
print('cuda devices:', ctranslate2.get_cuda_device_count())
assert int(av.__version__.split('.')[0]) < 19, 'PyAV 19+ tidak kompatibel; jalankan ulang install cell'
from faster_whisper import WhisperModel
m = WhisperModel(os.environ['WHISPER_MODEL'], device='cuda', compute_type=os.environ['WHISPER_COMPUTE'])
print('model loaded on cuda OK')
import tempfile, wave
with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
    w=wave.open(f.name,'wb'); w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000); w.writeframes(b'\x00\x00'*48000); w.close(); p=f.name
from faster_whisper.audio import decode_audio
audio = decode_audio(p, sampling_rate=16000)
print('decode ok, samples:', len(audio))
segs,_=m.transcribe(p, language='en'); print('smoke segments:', len(list(segs)))
import os; os.remove(p)


In [ ]:
# Tampilkan kredensial kapan saja (jalankan ulang cell ini)
import os
tok = open('/content/.clipper_token').read().strip() if os.path.exists('/content/.clipper_token') else '(belum ada)'
url = open('/content/.clipper_url').read().strip() if os.path.exists('/content/.clipper_url') else '(belum ada - jalankan cell tunnel)'
print('='*60)
print('TUNNEL URL :', url)
print('TOKEN      :', tok)
print('='*60)


In [ ]:
import glob, site
libdirs=[]
for sp in site.getsitepackages():
    libdirs += [d for d in glob.glob(sp + '/nvidia/*/lib') if __import__('os').path.isdir(d)]
import subprocess, time, re, threading
def run_worker():
    import os as _os
    _env = dict(_os.environ)
    _env['LD_LIBRARY_PATH'] = ':'.join(libdirs + [_env.get('LD_LIBRARY_PATH','')])
    subprocess.run(['uvicorn','worker:app','--host','0.0.0.0','--port','8010','--workers','1'], env=_env)
threading.Thread(target=run_worker, daemon=True).start()
time.sleep(12)
import urllib.request
print(urllib.request.urlopen('http://127.0.0.1:8010/health').read().decode())
print('TOKEN      :', open('/content/.clipper_token').read().strip())
proc = subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:8010'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for i,line in enumerate(proc.stdout):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        open('/content/.clipper_url','w').write(url)
        break
    if i > 2000:
        break
if url:
    print('='*60)
    print('TUNNEL URL :', url)
    print('TOKEN      :', open('/content/.clipper_token').read().strip())
    print('='*60)
else:
    print('URL tunnel tidak ditemukan. Cek output cloudflared di atas.')
    print('TOKEN (tetap bisa dipakai kalau URL sudah ada):', open('/content/.clipper_token').read().strip())
proc.wait()
